# 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import skew

DATA_DIR = Path("../data/raw/m5")
PROCESSED_DIR = Path("../data/processed")

SALES_PATH = DATA_DIR / "sales_train_validation.csv"
CALENDAR_PATH = PROCESSED_DIR / "calendar_cleaned.csv"

print("Stationarity & Decomposition Environment Ready!")
print("Sales File Exists:", SALES_PATH.exists())
print("Calendar File Exists:", CALENDAR_PATH.exists())

Stationarity & Decomposition Environment Ready!
Sales File Exists: True
Calendar File Exists: True


# 2. Load Daily Sales TimeSeries
We'll recreate the daily sales series from our M5 dataset so Notebook 05 works independently of Notebook 04.

In [2]:
daily_columns = [
    col for col in pd.read_csv(SALES_PATH, nrows=0).columns
    if col.startswith("d_")
]

daily_sales = np.zeros(
    len(daily_columns),
    dtype=np.int64
)

for chunk in pd.read_csv(SALES_PATH, chunksize=500):
    daily_sales += (
        chunk[daily_columns]
        .sum()
        .to_numpy(dtype=np.int64)
    )

calendar = pd.read_csv(CALENDAR_PATH, parse_dates=["date"])

daily_ts = pd.DataFrame({
    "date": calendar.set_index("d").loc[
        daily_columns, "date"
    ].values,
    "sales": daily_sales
})

daily_ts = daily_ts.sort_values("date").set_index("date")

print("Daily Time Series Shape:", daily_ts.shape)
print("\nDate Range:", daily_ts.index.min(), "to", daily_ts.index.max())
print("\nMissing Values:", daily_ts["sales"].isna().sum())
print("\nDaily Sales Summary:")
display(daily_ts["sales"].describe())

Daily Time Series Shape: (1913, 1)

Date Range: 2011-01-29 00:00:00 to 2016-04-24 00:00:00

Missing Values: 0

Daily Sales Summary:


count     1913.000000
mean     34341.562467
std       7345.905196
min         11.000000
25%      29654.000000
50%      33732.000000
75%      38689.000000
max      57218.000000
Name: sales, dtype: float64

One thing to note: our data ends on 24 April 2016. We'll use this actual date range throughout the analysis.

# 3. Additive vs Multiplicative Decomposition
First, we'll compare two decomposition methods.
- Additive: Seasonal variations remain roughly constant.
- Multiplicative: Seasonal variations change proportionally with the trend.

In [3]:
# Additive decomposition
additive_decomp = seasonal_decompose(
    daily_ts["sales"],
    model="additive",
    period=7
)

# Multiplicative decomposition
multiplicative_decomp = seasonal_decompose(
    daily_ts["sales"],
    model="multiplicative",
    period=7
)

# Compare seasonal variation
comparison = pd.DataFrame({
    "Additive Seasonal": additive_decomp.seasonal,
    "Multiplicative Seasonal": multiplicative_decomp.seasonal
})

print("Seasonal Component Comparison:")
display(comparison.describe())

print("\nAdditive Residual Standard Deviation:",
      additive_decomp.resid.std())

print("Multiplicative Residual Standard Deviation:",
      multiplicative_decomp.resid.std())

Seasonal Component Comparison:


,Additive Seasonal,Multiplicative Seasonal
count,1913.000000,1913.000000
mean,7.322115,1.000213
std,4661.712421,0.135999
min,-4324.895806,0.873911
25%,-4139.353682,0.878544
50%,-1459.929127,0.957713
75%,6789.470558,1.196267
max,7217.734738,1.211879



Additive Residual Standard Deviation: 2663.686042724987
Multiplicative Residual Standard Deviation: 0.08364470322375006


Interpretation
1. Additive decomposition
- Seasonal mean: 7.32 (almost zero).
- Seasonal standard deviation: 4,661.71.
- Seasonal values range from -4,324.90 to 7,217.73.
This represents seasonal changes in terms of actual sales units.

2. Multiplicative decomposition
- Seasonal mean: 1.0002 (almost 1).
- Seasonal standard deviation: 0.136.
- Seasonal values range from 0.874 to 1.212.
This represents seasonal changes as proportions of the trend.

3. Residual comparison
- Additive residual standard deviation: 2,663.69.
- Multiplicative residual standard deviation: 0.0836.
We cannot directly compare these two standard deviations because they're measured on different scales.
### Observations: Additive vs Multiplicative Decomposition

1. Additive decomposition represents seasonal variations in actual sales units.
2. Multiplicative decomposition represents seasonal variations as relative proportions.
3. The additive seasonal component has a mean close to zero.
4. The multiplicative seasonal component has a mean close to one.
5. Both methods identify weekly seasonal patterns in the sales series.

**Conclusion:** Both decomposition methods are mathematically suitable for this positive sales dataset. We will further compare their residual behaviour and seasonal variation before selecting the more appropriate approach for forecasting.

# 4. Compare Residual Behaviour
Purpose: Understand residual distribution, spread and skewness. This will help us decide which decomposition represents the sales pattern more appropriately.

In [4]:
additive_resid = additive_decomp.resid.dropna()
multiplicative_resid = multiplicative_decomp.resid.dropna()

residual_comparison = pd.DataFrame({
    "Additive Residual": additive_resid.describe(),
    "Multiplicative Residual": multiplicative_resid.describe()
})

display(residual_comparison)

print("\nAdditive Residual Skewness:",
      skew(additive_resid))

print("Multiplicative Residual Skewness:",
      skew(multiplicative_resid))

,Additive Residual,Multiplicative Residual
count,1907.000000,1907.000000
mean,0.546774,0.999711
std,2663.686043,0.083645
min,-31101.184843,0.000429
25%,-1172.986659,0.966823
50%,-86.707953,0.996999
75%,1107.240052,1.030877
max,13020.434904,1.399508



Additive Residual Skewness: -2.4776380363440373
Multiplicative Residual Skewness: -3.928761717497952


Interpretation
- Additive residuals: Mean is close to 0, but the large standard deviation and negative skewness indicate some unusually low residual values.
- Multiplicative residuals: Mean is close to 1, with negative skewness and some unusually low relative residuals.
- Both residual distributions are negatively skewed, suggesting that a few observations have pulled the residuals down substantially.
- The standard deviations are on different scales, so we shouldn't compare them directly
### Observations: Residual Comparison

1. Additive residuals have a mean close to zero.
2. Multiplicative residuals have a mean close to one.
3. Both residual distributions show negative skewness.
4. The negative skewness indicates the presence of unusually low residual values.
5. Residual standard deviations cannot be directly compared because the two decomposition methods use different scales.

Conclusion: Both decomposition methods leave some unexplained variation. Further residual diagnostics are required to examine whether autocorrelation remains.

# 5. Residual Autocorrelation Test
Now we'll use the Ljung–Box test to check whether residuals still contain autocorrelation.

In [5]:
from statsmodels.stats.diagnostic import acorr_ljungbox

additive_test = acorr_ljungbox(
    additive_resid,
    lags=[7, 14, 21],
    return_df=True
)

multiplicative_test = acorr_ljungbox(
    multiplicative_resid,
    lags=[7, 14, 21],
    return_df=True
)

print("Additive Residual Ljung-Box Test:")
display(additive_test)

print("\nMultiplicative Residual Ljung-Box Test:")
display(multiplicative_test)

Additive Residual Ljung-Box Test:


,lb_stat,lb_pvalue
7,364.185568,1.130407e-74
14,422.523340,2.260820e-81
21,457.336525,1.167072e-83



Multiplicative Residual Ljung-Box Test:


,lb_stat,lb_pvalue
7,281.879773,4.459749e-57
14,291.980415,5.545789e-54
21,300.622504,2.376208e-51


Interpretation
The Ljung–Box test checks whether residuals behave like random noise.
- Null hypothesis (H₀): No significant autocorrelation exists.
- Alternative hypothesis (H₁): Significant autocorrelation exists.
### Observations: Residual Autocorrelation

1. Both additive and multiplicative residuals show significant autocorrelation.
2. All Ljung-Box test p-values are below 0.05.
3. The residuals do not behave like independent random noise.
4. Some time-dependent patterns remain unexplained after decomposition.

Conclusion: Decomposition alone is insufficient to capture all patterns in the sales series. ARIMA/SARIMA models will be explored to capture the remaining autocorrelation.

# 6. Log Transformation
Now let's check whether a log transformation helps stabilize the variation in sales.

In [6]:
log_sales = np.log(daily_ts["sales"])

print("Original Sales Skewness:", skew(daily_ts["sales"]))
print("Log Transformed Sales Skewness:", skew(log_sales))

print("\nOriginal Sales Standard Deviation:",
      daily_ts["sales"].std())

print("Log Transformed Sales Standard Deviation:",
      log_sales.std())

Original Sales Skewness: 0.03716936228208205
Log Transformed Sales Skewness: -13.364305859840728

Original Sales Standard Deviation: 7345.905196022218
Log Transformed Sales Standard Deviation: 0.4466336095938989


Interpretation
- Original sales skewness is close to zero, indicating an approximately symmetric distribution.
- Log-transformed sales have strong negative skewness.
- Standard deviations are on different scales, so they cannot be compared directly.
- This suggests log transformation may not be beneficial for this dataset.
One important point: the unusually low sales observations (especially holiday-related drops) may be contributing to the negative skewness after transformation.
### Observations: Log Transformation

1. Original sales data has skewness close to zero (0.037).
2. Log transformation produces strong negative skewness (-13.365).
3. The transformed distribution is more negatively skewed than the original distribution.
4. Log transformation does not appear beneficial for improving distribution symmetry in this dataset.

Conclusion: We will retain the original sales scale for further analysis unless model evaluation demonstrates an advantage from transformation.

# 7. Compare Rolling Variation
Instead of judging transformation using standard deviation alone, let's compare the coefficient of variation across different time periods.

In [7]:
comparison_df = pd.DataFrame({
    "Original Sales": daily_ts["sales"],
    "Log Sales": log_sales
})

rolling_comparison = pd.DataFrame({
    "Original Rolling Mean": comparison_df["Original Sales"].rolling(30).mean(),
    "Original Rolling Std": comparison_df["Original Sales"].rolling(30).std(),
    "Log Rolling Mean": comparison_df["Log Sales"].rolling(30).mean(),
    "Log Rolling Std": comparison_df["Log Sales"].rolling(30).std()
})

print("Original Sales Rolling CV:")
display(
    (rolling_comparison["Original Rolling Std"] /
     rolling_comparison["Original Rolling Mean"]).describe()
)

print("\nLog Sales Rolling Standard Deviation:")
display(rolling_comparison["Log Rolling Std"].describe())

Original Sales Rolling CV:


count    1884.000000
mean        0.167875
std         0.027573
min         0.114802
25%         0.149698
50%         0.165279
75%         0.178437
max         0.259044
dtype: float64


Log Sales Rolling Standard Deviation:


count    1884.000000
mean        0.258641
std         0.340230
min         0.113519
25%         0.146538
50%         0.161771
75%         0.174925
max         1.476588
Name: Log Rolling Std, dtype: float64

Interpretation
Original Sales — Rolling CV
- Mean: 0.168
- Median: 0.165
- Maximum: 0.259
Log Sales — Rolling Standard Deviation
- Mean: 0.259
- Median: 0.162
- Maximum: 1.477
The log rolling standard deviation has a much larger maximum and variability (std = 0.340), suggesting some periods still have substantial variation.

However, CV and log rolling standard deviation are not directly comparable. We shouldn't declare a winner from these numbers alone.
### Observations: Transformation Comparison

1. The original sales rolling coefficient of variation has a mean of 0.168.
2. The log-transformed series has a rolling standard deviation mean of 0.259.
3. Both measures show variation across different time periods.
4. Log transformation does not provide clear evidence of improved variance stability.
5. The original sales series will be retained as the primary series for forecasting.

Conclusion: Based on the current analysis, log transformation is not clearly beneficial. Final transformation decisions will also consider forecasting performance.

# 8. Final Stationarity Check
checking the ADF test on the log-transformed series. Remember, log transformation and differencing serve different purposes.

In [8]:
log_adf_result = adfuller(log_sales)

print("Log Transformed Series ADF Statistic:",
      log_adf_result[0])

print("Log Transformed Series p-value:",
      log_adf_result[1])

print("Critical Values:")
for key, value in log_adf_result[4].items():
    print(f"{key}: {value}")

Log Transformed Series ADF Statistic: -4.270386152334241
Log Transformed Series p-value: 0.0004996931056778759
Critical Values:
1%: -3.433822016251123
5%: -2.8630736989499335
10%: -2.567586484303047


/tmp/ipykernel_6025/2094964236.py:1: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  log_adf_result = adfuller(log_sales)


Interpretation: Log ADF Test
Stationary
- ADF Statistic: -4.2704
- p-value: 0.0005
- 1% Critical Value: -3.4338
Since the p-value is below 0.05 and the ADF statistic is lower than even the 1% critical value, we reject the null hypothesis of a unit root.

One important distinction: ADF tests for a unit root; it does not guarantee that every form of trend, seasonality or autocorrelation has disappeared.
### Final Findings: Stationarity & Decomposition

1. Both additive and multiplicative decomposition identify weekly seasonality.
2. Residual diagnostics show significant autocorrelation in both methods.
3. The residual distributions exhibit negative skewness.
4. Log transformation changes the distribution substantially and produces strong negative skewness.
5. The log-transformed series passes the ADF stationarity test with a p-value of 0.0005.
6. Log transformation does not provide clear evidence of improved variance stability.

Conclusion: The original sales series will remain our primary forecasting series. However, the log-transformed series may be considered as an alternative during model evaluation. Further modelling is required to capture the remaining autocorrelation and seasonal patterns.